# GRACE-FO THM 计算（MeTp，解析法，D 发射 C 接收）

这版程序按你的更正实现 **MeTp 的 THM**，其中：

- **发射端：D 星**
- **接收端：C 星**
- **T → C 星**
- **M → D 星**

采用的展开式为：

\[
\mathbf r_r = \mathbf r_C\!\left(t_r-\Delta t_{\mathrm{TpMr}}\right)
\]

\[
\mathbf r_e = \mathbf r_D\!\left(t_r-\Delta t_{\mathrm{TpMr}}-\Delta t_{\mathrm{MeTp}}\right)
\]

其中

并且方向向量按最新更正取为：

\[
\mathbf d_0=\frac{\mathbf r_M-\mathbf r_T}{\left|\mathbf r_M-\mathbf r_T\right|}
\]

在当前代码映射下即：

\[
\mathbf d_0=\frac{\mathbf r_D-\mathbf r_C}{\left|\mathbf r_D-\mathbf r_C\right|}
\]


\[
\Delta t_{\mathrm{TpMr}} = \Delta t_{\mathrm{inst}}\left(1+\frac{\mathbf d_0\!\cdot\!\mathbf v_T}{c_0}\right)
\]

\[
\Delta t_{\mathrm{MeTp}} = \Delta t_{\mathrm{inst}}\,\frac{2c_0+\mathbf d_0\!\cdot\!\mathbf v_T-\mathbf d_0\!\cdot\!\mathbf v_M}{c_0}
\]

并按你的上一次更正，发射端展开时间取

\[
\Delta t_{\mathrm{emit}}=\Delta t_{\mathrm{TpMr}}+\Delta t_{\mathrm{MeTp}}
\]

因此二阶泰勒展开为：

\[
\mathbf r_r \approx \mathbf r_C(t_r)-\mathbf v_C(t_r)\,\Delta t_{\mathrm{TpMr}}+\frac12\,\mathbf a_C(t_r)\,\Delta t_{\mathrm{TpMr}}^2
\]

\[
\mathbf r_e \approx \mathbf r_D(t_r)-\mathbf v_D(t_r)\,(\Delta t_{\mathrm{TpMr}}+\Delta t_{\mathrm{MeTp}})
+\frac12\,\mathbf a_D(t_r)\,(\Delta t_{\mathrm{TpMr}}+\Delta t_{\mathrm{MeTp}})^2
\]

加速度按地球点质量模型：

\[
\mathbf a = -\frac{GM}{R^3}\,\mathbf r,
\qquad
|\mathbf a| = \frac{GM}{R^2}
\]

运行结束后会输出 Excel 文件。


本版本已改成 **兼容旧版 Python 内核** 的类型注解写法。


另外，`T_HM` 的光路积分部分已改为按论文中的**分段梯形积分**方式显式求和，而不是直接调用整体积分函数。


这版把 `T_HM` 的光路积分改成按论文式 (4.21) 的**端点半权、内部点全权**形式计算：
\[
T_{HM}^{(N-1)}
pprox
rac{2\,\Delta t_{SR}}{c_0^2\,N}
\left(
\sum_{n=1}^{N-1} W_{HM}(	ilde t_n,ec r_{ph}(\lambda_n))
+rac{W_{HM}(	ilde t_N,ec r_{ph}(\lambda_N))+W_{HM}(	ilde t_0,ec r_{ph}(\lambda_0))}{2}

ight)
\]


In [15]:

from __future__ import annotations

from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Dict, List, Tuple, Union

import numpy as np
import pandas as pd
from scipy.integrate import trapezoid
from scipy.special import gammaln, lpmv

import astropy.units as u
from astropy.coordinates import CartesianRepresentation, GCRS, ITRS
from astropy.time import Time, TimeDelta
from astropy.utils import iers

try:
    from tqdm import tqdm
except Exception:
    tqdm = None

try:
    iers.conf.auto_download = True
except Exception:
    pass


CONFIG = {
    # 输入文件
    "c_file": r"GNI1B_2022-06-05_C_04.txt",   # C 星
    "d_file": r"GNI1B_2022-06-05_D_04.txt",   # D 星
    "gfc_file": r"EIGEN-6C4.gfc",             # 静力场模型

    # THM 参数
    "lmax": 2,
    "n_path": 10,
    "n_potential_path": 15,
    "r_ref_factor": 50.0,
    "max_rows": None,
    "show_progress": True,

    # 输出文件
    "out_xlsx": "MeTp_THM_D_emit_C_recv_taylor2_formula_emit_sum_d0_MD_minus_TC_eq421_pyold_minimal.xlsx",
}

# 常数
C0 = 299_792_458.0
G_CONST = 6.67430e-11
M_EARTH = 5.97219e24
GM_POINT_MASS = G_CONST * M_EARTH
GNI_GPS_TIME_EPOCH_UTC = Time("2000-01-01T11:59:47", scale="utc")


@dataclass
class GravityModel:
    GM: float
    Re: float
    C: np.ndarray
    S: np.ndarray
    lmax_use: int
    signed_norm_table: np.ndarray
    m_index: np.ndarray


def norm3(x: np.ndarray) -> float:
    return float(np.linalg.norm(np.asarray(x, dtype=float)))


def unit3(x: np.ndarray, eps: float = 1e-30) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    n = norm3(x)
    if n < eps:
        raise ValueError("向量范数过小，无法单位化。")
    return x / n


def point_mass_acceleration(r_xyz_m: np.ndarray, gm_value: float = GM_POINT_MASS) -> np.ndarray:
    r_xyz_m = np.asarray(r_xyz_m, dtype=float)
    r_norm = norm3(r_xyz_m)
    if r_norm < 1e-6:
        raise ValueError("位置向量过小，无法计算点质量加速度。")
    return -(gm_value / r_norm**3) * r_xyz_m


def second_order_taylor_back_propagation(
    r_tr: np.ndarray,
    v_tr: np.ndarray,
    a_tr: np.ndarray,
    dt: float,
) -> np.ndarray:
    r_tr = np.asarray(r_tr, dtype=float)
    v_tr = np.asarray(v_tr, dtype=float)
    a_tr = np.asarray(a_tr, dtype=float)
    dt = float(dt)
    return r_tr - v_tr * dt + 0.5 * a_tr * dt**2


def gps_seconds_to_astropy_time(gps_seconds_from_2000_epoch: float) -> Time:
    return GNI_GPS_TIME_EPOCH_UTC + TimeDelta(float(gps_seconds_from_2000_epoch), format="sec")


def read_gni1b_txt(path: Union[str, Path]) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"未找到文件: {path}")

    lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()
    try:
        end_idx = next(i for i, line in enumerate(lines) if line.strip() == "# End of YAML header")
    except StopIteration as exc:
        raise ValueError(f"文件 {path} 未找到 '# End of YAML header' 标记。") from exc

    col_names = [
        "gps_time", "sat_id", "coord_ref",
        "xpos", "ypos", "zpos",
        "xpos_err", "ypos_err", "zpos_err",
        "xvel", "yvel", "zvel",
        "xvel_err", "yvel_err", "zvel_err",
        "qualflg",
    ]

    df = pd.read_csv(
        path,
        sep=r"\s+",
        skiprows=end_idx + 1,
        header=None,
        names=col_names,
        engine="python",
    )

    df = df[["gps_time", "sat_id", "coord_ref", "xpos", "ypos", "zpos", "xvel", "yvel", "zvel"]].copy()
    if not (df["coord_ref"] == "I").all():
        raise ValueError(f"文件 {path} 含非惯性系记录，coord_ref 应为 'I'。")
    return df.reset_index(drop=True)


def prepare_satellite_dataframe(path: Union[str, Path]) -> pd.DataFrame:
    return read_gni1b_txt(path)


def build_signed_norm_table(lmax: int) -> np.ndarray:
    signed_norm = np.zeros((lmax + 1, lmax + 1), dtype=float)
    for l in range(lmax + 1):
        for m in range(l + 1):
            delta_m0 = 1.0 if m == 0 else 0.0
            logN = 0.5 * (
                np.log((2.0 - delta_m0) * (2 * l + 1))
                + gammaln(l - m + 1)
                - gammaln(l + m + 1)
            )
            signed_norm[l, m] = ((-1.0) ** m) * np.exp(logN)
    return signed_norm


def load_icgem_gfc(path: Union[str, Path], lmax_use: int) -> GravityModel:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"未找到 gfc 文件: {path}")

    GM = None
    Re = None
    max_degree_file = None
    norm_type = None
    coeffs: List[Tuple[int, int, float, float]] = []

    in_header = True
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if in_header:
                if line.startswith("end_of_head"):
                    in_header = False
                    continue
                parts = line.split()
                key = parts[0]
                if key == "earth_gravity_constant":
                    GM = float(parts[1])
                elif key == "radius":
                    Re = float(parts[1])
                elif key == "max_degree":
                    max_degree_file = int(parts[1])
                elif key == "norm":
                    norm_type = parts[1]
                continue

            if line.startswith("gfc"):
                parts = line.split()
                l = int(parts[1])
                m = int(parts[2])
                if l <= lmax_use:
                    coeffs.append((l, m, float(parts[3]), float(parts[4])))

    if GM is None or Re is None or max_degree_file is None:
        raise ValueError("gfc 文件头读取失败，请检查文件格式。")
    if lmax_use > max_degree_file:
        raise ValueError(f"lmax_use={lmax_use} 超过文件最大阶数 {max_degree_file}")
    if str(norm_type).lower() != "fully_normalized":
        raise ValueError(f"当前代码按 fully_normalized 系数实现，文件 norm={norm_type} 需匹配。")

    C = np.zeros((lmax_use + 1, lmax_use + 1), dtype=float)
    S = np.zeros((lmax_use + 1, lmax_use + 1), dtype=float)
    for l, m, c, s in coeffs:
        C[l, m] = c
        S[l, m] = s

    return GravityModel(
        GM=GM,
        Re=Re,
        C=C,
        S=S,
        lmax_use=lmax_use,
        signed_norm_table=build_signed_norm_table(lmax_use),
        m_index=np.arange(lmax_use + 1, dtype=float),
    )


@lru_cache(maxsize=4096)
def gcrs_to_itrs_rotation_matrix(gps_time_s: float) -> np.ndarray:
    gps_time_s = float(gps_time_s)
    obstime = gps_seconds_to_astropy_time(gps_time_s)
    basis = np.eye(3, dtype=float)
    mat = np.empty((3, 3), dtype=float)
    for j in range(3):
        rep = CartesianRepresentation(x=basis[0, j] * u.m, y=basis[1, j] * u.m, z=basis[2, j] * u.m)
        gcrs = GCRS(rep, obstime=obstime)
        itrs = gcrs.transform_to(ITRS(obstime=obstime))
        mat[:, j] = np.array(itrs.cartesian.xyz.to_value(u.m), dtype=float)
    return mat


def cartesian_to_spherical(r_xyz: np.ndarray) -> Tuple[float, float, float]:
    x, y, z = np.asarray(r_xyz, dtype=float)
    r = norm3(r_xyz)
    if r == 0.0:
        raise ValueError("位置向量为零，无法转球坐标。")
    theta = np.arccos(np.clip(z / r, -1.0, 1.0))
    lamb = np.arctan2(y, x)
    return r, theta, lamb


def build_fully_normalized_plm_tables(
    lmax: int,
    x: float,
    theta: float,
    signed_norm_table: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray]:
    pbar = np.zeros((lmax + 1, lmax + 1), dtype=float)
    dpbar_dtheta = np.zeros((lmax + 1, lmax + 1), dtype=float)

    for l in range(lmax + 1):
        m_arr = np.arange(l + 1)
        pbar[l, :l + 1] = signed_norm_table[l, :l + 1] * lpmv(m_arr, l, x)

    sin_theta = float(np.sin(theta))
    if abs(sin_theta) < 1e-15:
        sin_theta = 1e-15 if sin_theta >= 0.0 else -1e-15

    for l in range(1, lmax + 1):
        for m in range(l + 1):
            if l - 1 >= m:
                coeff = np.sqrt(((2.0 * l + 1.0) / (2.0 * l - 1.0)) * (l * l - m * m))
                pbar_lm1 = pbar[l - 1, m]
            else:
                coeff = 0.0
                pbar_lm1 = 0.0
            dpbar_dtheta[l, m] = (l * x * pbar[l, m] - coeff * pbar_lm1) / sin_theta

    return pbar, dpbar_dtheta


def compute_static_whm_from_itrs(r_itrs_m: np.ndarray, gravity: GravityModel) -> float:
    r, theta, lamb = cartesian_to_spherical(r_itrs_m)
    x = np.cos(theta)

    pbar, _ = build_fully_normalized_plm_tables(
        gravity.lmax_use, x, theta, gravity.signed_norm_table
    )

    m_index = gravity.m_index
    cos_m_lambda = np.cos(m_index * lamb)
    sin_m_lambda = np.sin(m_index * lamb)

    q = gravity.Re / r
    radial = q * q
    total = 0.0

    for l in range(1, gravity.lmax_use + 1):
        trig = gravity.C[l, :l + 1] * cos_m_lambda[:l + 1] + gravity.S[l, :l + 1] * sin_m_lambda[:l + 1]
        total += radial * float(np.dot(trig, pbar[l, :l + 1]))
        radial *= q

    return gravity.GM / gravity.Re * total


def compute_static_ahm_from_itrs(r_itrs_m: np.ndarray, gravity: GravityModel) -> np.ndarray:
    r, theta, lamb = cartesian_to_spherical(r_itrs_m)
    x = np.cos(theta)
    sin_theta = float(np.sin(theta))
    if abs(sin_theta) < 1e-15:
        sin_theta = 1e-15 if sin_theta >= 0.0 else -1e-15

    pbar, dpbar_dtheta = build_fully_normalized_plm_tables(
        gravity.lmax_use, x, theta, gravity.signed_norm_table
    )

    m_index = gravity.m_index
    cos_m_lambda = np.cos(m_index * lamb)
    sin_m_lambda = np.sin(m_index * lamb)

    dW_dr = 0.0
    dW_dtheta = 0.0
    dW_dlambda = 0.0

    q = gravity.Re / r
    radial = q * q

    for l in range(1, gravity.lmax_use + 1):
        pref = gravity.GM / gravity.Re * radial

        trig = gravity.C[l, :l + 1] * cos_m_lambda[:l + 1] + gravity.S[l, :l + 1] * sin_m_lambda[:l + 1]
        dtrig_dlambda = m_index[:l + 1] * (
            -gravity.C[l, :l + 1] * sin_m_lambda[:l + 1] + gravity.S[l, :l + 1] * cos_m_lambda[:l + 1]
        )

        pbar_l = pbar[l, :l + 1]
        dpbar_l = dpbar_dtheta[l, :l + 1]

        dW_dr += pref * (-(l + 1) / r) * float(np.dot(trig, pbar_l))
        dW_dtheta += pref * float(np.dot(trig, dpbar_l))
        dW_dlambda += pref * float(np.dot(dtrig_dlambda, pbar_l))

        radial *= q

    a_r = dW_dr
    a_theta = dW_dtheta / r
    a_lambda = dW_dlambda / (r * sin_theta)

    st = np.sin(theta)
    ct = np.cos(theta)
    cl = np.cos(lamb)
    sl = np.sin(lamb)

    e_r = np.array([st * cl, st * sl, ct], dtype=float)
    e_theta = np.array([ct * cl, ct * sl, -st], dtype=float)
    e_lambda = np.array([-sl, cl, 0.0], dtype=float)

    return a_r * e_r + a_theta * e_theta + a_lambda * e_lambda


def compute_static_whm_from_gcrs_line_integral(
    gps_time_s: float,
    r_gcrs_m: np.ndarray,
    gravity: GravityModel,
    n_potential_path: int = 64,
    r_ref_factor: float = 50.0,
) -> float:
    if n_potential_path < 2:
        raise ValueError("n_potential_path 至少应为 2。")
    if r_ref_factor <= 1.0:
        raise ValueError("r_ref_factor 应大于 1。")

    gps_time_s = float(gps_time_s)
    r_gcrs_m = np.asarray(r_gcrs_m, dtype=float)
    r0 = norm3(r_gcrs_m)
    u_hat = unit3(r_gcrs_m)
    r_ref = max(float(r_ref_factor) * gravity.Re, r0 * (1.0 + 1e-12))

    rho_nodes = np.geomspace(r0, r_ref, int(n_potential_path) + 1)
    a_dot_dr = np.empty_like(rho_nodes)

    rot_g2i = gcrs_to_itrs_rotation_matrix(gps_time_s)
    u_hat_itrs = rot_g2i @ u_hat

    for i, rho in enumerate(rho_nodes):
        point_i_itrs = u_hat_itrs * rho
        a_itrs_i = compute_static_ahm_from_itrs(point_i_itrs, gravity)
        a_dot_dr[i] = float(np.dot(a_itrs_i, u_hat_itrs))

    r_ref_itrs = u_hat_itrs * r_ref
    w_ref = compute_static_whm_from_itrs(r_ref_itrs, gravity)
    return float(w_ref - trapezoid(a_dot_dr, x=rho_nodes))


def evaluate_whm_at_gcrs(
    gps_time_s: float,
    r_gcrs_m: np.ndarray,
    gravity: GravityModel,
    n_potential_path: int = 64,
    r_ref_factor: float = 50.0,
) -> float:
    return compute_static_whm_from_gcrs_line_integral(
        gps_time_s=float(gps_time_s),
        r_gcrs_m=r_gcrs_m,
        gravity=gravity,
        n_potential_path=n_potential_path,
        r_ref_factor=r_ref_factor,
    )


def compute_thm(
    te_seconds: float,
    delta_t_sr: float,
    re_gcrs: np.ndarray,
    rr_gcrs: np.ndarray,
    gravity: GravityModel,
    n_path: int = 16,
    n_potential_path: int = 64,
    r_ref_factor: float = 50.0,
) -> float:
    """
    按论文式 (4.21) 的分段梯形积分形式计算 T_HM：

        T_HM^(N-1) ≈ 2 * Δt_SR / (c0^2 * N)
                     * [ Σ_{n=1}^{N-1} W_n + (W_0 + W_N)/2 ]

    这里取：
        λ_n = n / N,  n = 0, 1, ..., N

    并在每个节点上计算：
        t_n = t_e + λ_n * Δt_SR
        r_n = r_e + λ_n * (r_r - r_e)

    注意：
    - 这与显式逐段梯形求和在等步长情况下是完全等价的；
    - 这里直接写成论文里的“端点半权、内部点全权”形式。
    """
    if n_path < 1:
        raise ValueError("n_path 至少应为 1。")

    N = int(n_path)

    lam = np.linspace(0.0, 1.0, N + 1)
    t_nodes = te_seconds + delta_t_sr * lam
    r_nodes = re_gcrs[None, :] + (rr_gcrs - re_gcrs)[None, :] * lam[:, None]

    w_vals = np.empty(N + 1, dtype=float)
    for i in range(N + 1):
        w_vals[i] = evaluate_whm_at_gcrs(
            gps_time_s=float(t_nodes[i]),
            r_gcrs_m=r_nodes[i],
            gravity=gravity,
            n_potential_path=n_potential_path,
            r_ref_factor=r_ref_factor,
        )

    if N == 1:
        weighted_sum = 0.5 * (float(w_vals[0]) + float(w_vals[1]))
    else:
        weighted_sum = float(np.sum(w_vals[1:N])) + 0.5 * (float(w_vals[0]) + float(w_vals[N]))

    return 2.0 * float(delta_t_sr) / (C0**2 * N) * weighted_sum



def compute_metp_thm_formula_analytic(cfg: Dict[str, object]) -> pd.DataFrame:
    """
    按最新更正后的展开式计算 MeTp 的 THM：
        接收端: C 星
        发射端: D 星

        r_r = r_C(tr - ΔtTpMr)
        r_e = r_D(tr - ΔtTpMr - ΔtMeTp)

        ΔtTpMr = Δtinst * (1 + d0·v_T / c0),  其中 T -> C 星，且 d0 = (r_M - r_T)/|r_M - r_T|
        ΔtMeTp = Δtinst * (2c0 + d0·v_T - d0·v_M) / c0, 其中 M -> D 星

    并按用户更正，发射端二阶展开时间取:
        Δt_emit_total = ΔtTpMr + ΔtMeTp
    """
    gravity = load_icgem_gfc(str(cfg["gfc_file"]), lmax_use=int(cfg.get("lmax", 20)))

    c_df = prepare_satellite_dataframe(str(cfg["c_file"]))
    d_df = prepare_satellite_dataframe(str(cfg["d_file"]))

    merged = pd.merge(
        c_df,
        d_df,
        on="gps_time",
        suffixes=("_C", "_D"),
        how="inner",
    ).sort_values("gps_time").reset_index(drop=True)

    max_rows = cfg.get("max_rows", None)
    if max_rows not in [None, "", "None"]:
        merged = merged.iloc[:int(max_rows)].copy()

    records = merged[[
        "gps_time",
        "xpos_C", "ypos_C", "zpos_C",
        "xvel_C", "yvel_C", "zvel_C",
        "xpos_D", "ypos_D", "zpos_D",
        "xvel_D", "yvel_D", "zvel_D",
    ]].itertuples(index=False, name=None)

    total_rows = len(merged)
    show_progress = bool(cfg.get("show_progress", True))
    if show_progress and tqdm is not None:
        records = tqdm(records, total=total_rows, desc="MeTp THM 解析法计算进度")
    elif show_progress:
        print("MeTp THM 解析法计算开始 ...")

    n_path = int(cfg.get("n_path", 10))
    n_potential_path = int(cfg.get("n_potential_path", 20))
    r_ref_factor = float(cfg.get("r_ref_factor", 50.0))

    rows = []
    append_row = rows.append

    for (
        gps_time,
        xpos_C, ypos_C, zpos_C,
        xvel_C, yvel_C, zvel_C,
        xpos_D, ypos_D, zpos_D,
        xvel_D, yvel_D, zvel_D,
    ) in records:
        tr_seconds = float(gps_time)

        rC_tr = np.array([xpos_C, ypos_C, zpos_C], dtype=float)
        vC_tr = np.array([xvel_C, yvel_C, zvel_C], dtype=float)
        aC_tr = point_mass_acceleration(rC_tr)

        rD_tr = np.array([xpos_D, ypos_D, zpos_D], dtype=float)
        vD_tr = np.array([xvel_D, yvel_D, zvel_D], dtype=float)
        aD_tr = point_mass_acceleration(rD_tr)

        # 方向向量按用户最新更正：d0 = (r_M - r_T)/|r_M - r_T| = (r_D - r_C)/|r_D - r_C|
        d0_vec = rD_tr - rC_tr
        d0_hat = unit3(d0_vec)
        dt_inst = norm3(d0_vec) / C0

        # T -> C 星, M -> D 星
        d0_dot_vT = float(np.dot(d0_hat, vC_tr))
        d0_dot_vM = float(np.dot(d0_hat, vD_tr))

        dt_tpmr = dt_inst * (1.0 + d0_dot_vT / C0)
        dt_metp = dt_inst * ((2.0 * C0 + d0_dot_vT - d0_dot_vM) / C0)

        dt_n = dt_metp - dt_tpmr

        rr_gcrs = second_order_taylor_back_propagation(
            r_tr=rC_tr,
            v_tr=vC_tr,
            a_tr=aC_tr,
            dt=dt_tpmr,
        )
        re_gcrs = second_order_taylor_back_propagation(
            r_tr=rD_tr,
            v_tr=vD_tr,
            a_tr=aD_tr,
            dt=dt_metp,
        )

        tr_eff = tr_seconds - dt_tpmr
        te_eff = tr_seconds - dt_metp
        delta_t_sr = abs(tr_eff - te_eff)

        thm = compute_thm(
            te_seconds=te_eff,
            delta_t_sr=delta_t_sr,
            re_gcrs=re_gcrs,
            rr_gcrs=rr_gcrs,
            gravity=gravity,
            n_path=n_path,
            n_potential_path=n_potential_path,
            r_ref_factor=r_ref_factor,
        )

        append_row({
            "gps_time": tr_seconds,
            "T_HM": float(thm),
            "dt_inst_s": dt_inst,
            "dt_TpMr_s": dt_tpmr,
            "dt_MeTp_s": dt_metp,
            "dt_n_s": dt_n,
            "d0_x": d0_hat[0],
            "d0_y": d0_hat[1],
            "d0_z": d0_hat[2],
            "d0_dot_vT_mps": d0_dot_vT,
            "d0_dot_vM_mps": d0_dot_vM,
            "aC_x_mps2": aC_tr[0],
            "aC_y_mps2": aC_tr[1],
            "aC_z_mps2": aC_tr[2],
            "aC_mag_mps2": norm3(aC_tr),
            "aD_x_mps2": aD_tr[0],
            "aD_y_mps2": aD_tr[1],
            "aD_z_mps2": aD_tr[2],
            "aD_mag_mps2": norm3(aD_tr),
        })

    return pd.DataFrame(rows)


In [16]:
THM_DF = compute_metp_thm_formula_analytic(CONFIG)

# 只保留两列输出
THM_DF = THM_DF[["gps_time", "T_HM"]].rename(columns={"T_HM": "THM"})

THM_DF.to_excel(CONFIG["out_xlsx"], sheet_name="THM", index=False)

print(f"结果已写出: {CONFIG['out_xlsx']}")
THM_DF.head()


MeTp THM 解析法计算进度: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 86400/86400 [2:00:28<00:00, 11.95it/s]


结果已写出: MeTp_THM_D_emit_C_recv_taylor2_formula_emit_sum_d0_MD_minus_TC_pyold.xlsx


,gps_time,T_HM,dt_inst_s,dt_TpMr_s,dt_MeTp_s,dt_n_s,d0_x,d0_y,d0_z,d0_dot_vT_mps,d0_dot_vM_mps,aC_x_mps2,aC_y_mps2,aC_z_mps2,aC_mag_mps2,aD_x_mps2,aD_y_mps2,aD_z_mps2,aD_mag_mps2
0,707659200.0,-1.831796e-16,0.00065,0.00065,0.001301,0.00065,0.534905,0.433562,-0.725190,-7633.447095,-7633.454466,-4.603397,-3.973712,-5.921858,8.488230,-4.732041,-4.077956,-5.746041,8.487574
1,707659201.0,-1.846463e-16,0.00065,0.00065,0.001301,0.00065,0.535515,0.434087,-0.724425,-7633.446221,-7633.454250,-4.598274,-3.969574,-5.928644,8.488254,-4.727069,-4.073946,-5.753012,8.487600
2,707659202.0,-1.861132e-16,0.00065,0.00065,0.001301,0.00065,0.536126,0.434612,-0.723659,-7633.445323,-7633.454009,-4.593146,-3.965430,-5.935423,8.488279,-4.722090,-4.069931,-5.759976,8.487626
3,707659203.0,-1.875802e-16,0.00065,0.00065,0.001301,0.00065,0.536735,0.435136,-0.722892,-7633.444399,-7633.453743,-4.588012,-3.961282,-5.942195,8.488304,-4.717106,-4.065911,-5.766933,8.487652
4,707659204.0,-1.890474e-16,0.00065,0.00065,0.001301,0.00065,0.537344,0.435659,-0.722124,-7633.443450,-7633.453451,-4.582873,-3.957129,-5.948960,8.488328,-4.712116,-4.061887,-5.773883,8.487678
